# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Two signals to check first, from the session's real flags:

The value of days_since_last_update refers to the signal that causes the refresh flag to be updated.
The signal for the low-CTR / CTR-fix flag is the CTR when considering the pages ranked in the top 20 and actually visible.
Before allowing either one to determine the score, I check both of them against the is_declining_label (which is defined in the same way as the session did: trend_direction == 'down').

In [ ]:
import pandas as pd
import os

local_path = "../../data/raw/content_refresh_anonymized.csv"

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    if not os.path.exists("internship"):
        !git clone https://github.com/UnzilaAhsan/internship.git
    df = pd.read_csv("internship/data/raw/content_refresh_anonymized.csv")

# Same prep as the session's script: real visibility, not brand-new, one row per content item.
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} rows | base rate (declining): {df['is_declining_label'].mean():.3f}")

# --- Signal check 1: staleness -> refresh flag ---
bins   = [-1, 30, 90, 180, 365, 100000]
labels = ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)
sig1 = df.groupby("staleness_bucket", observed=True)["is_declining_label"].agg(["mean", "count"])
print("\nSIGNAL 1 — staleness vs decline rate (n printed):")
print(sig1)
print("VERDICT: MIXED — the two large buckets do move the right way (0-30d: "
      f"{sig1.loc['0-30d','mean']:.3f}, n={int(sig1.loc['0-30d','count'])} vs 91-180d: "
      f"{sig1.loc['91-180d','mean']:.3f}, n={int(sig1.loc['91-180d','count'])}), but the small-n "
      "buckets in between don't cooperate, and 181-365d actually dips below 0-30d. Not clean "
      "enough to anchor a score on alone.")

# --- Signal check 2: CTR-vs-position -> CTR-fix flag ---
good_pos = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 100)].copy()
ctr_bins   = [-1, 0.5, 1, 2, 5, 1000]
ctr_labels = ["<0.5%", "0.5-1%", "1-2%", "2-5%", "5%+"]
good_pos["ctr_bucket"] = pd.cut(good_pos["ctr"], bins=ctr_bins, labels=ctr_labels)
sig2 = good_pos.groupby("ctr_bucket", observed=True)["is_declining_label"].agg(["mean", "count"])
print(f"\nSIGNAL 2 — CTR vs decline rate, among top-20 visible pages (n={len(good_pos):,}):")
print(sig2)
print("VERDICT: CONFIRMED — decline rate falls almost monotonically as CTR rises "
      f"({sig2.loc['<0.5%','mean']:.3f} at <0.5% CTR down to {sig2.loc['2-5%','mean']:.3f} at "
      "2-5% CTR), across buckets with real n. This is the signal my rule will lean on.")


30,000 rows | base rate (declining): 0.542

SIGNAL 1 — staleness vs decline rate (n printed):
                      mean  count
staleness_bucket                 
0-30d             0.511377  20480
31-90d            0.588571    175
91-180d           0.611057   9171
181-365d          0.467456    169
365d+             0.600000      5
VERDICT: MIXED — the two large buckets do move the right way (0-30d: 0.511, n=20480 vs 91-180d: 0.611, n=9171), but the small-n buckets in between don't cooperate, and 181-365d actually dips below 0-30d. Not clean enough to anchor a score on alone.

SIGNAL 2 — CTR vs decline rate, among top-20 visible pages (n=15,091):
                mean  count
ctr_bucket                 
<0.5%       0.646890  12203
0.5-1%      0.510608   2121
1-2%        0.493130    655
2-5%        0.402062     97
5%+         0.533333     15
VERDICT: CONFIRMED — decline rate falls almost monotonically as CTR rises (0.647 at <0.5% CTR down to 0.402 at 2-5% CTR), across buckets with real n. T

Rule: Pages that can be marked for CTR-oriented optimization include pages with search visibility that rank high enough to get any visibility (rank within 20), but convert it into clicks very poorly, because CTR/position relation is the only signal that I proved to track decline for sure. I don’t use staleness as a basis for building a score, because my personal check of it resulted in inconsistency, and I don’t want to use the assumption that is already checked and didn’t prove to be true.

Reason Code: low_ctr_visible_page Action Label: refresh_and_review_ctr for the top decile by score, review otherwise.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
def percentile_rank(s):
    return s.rank(pct=True)

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["position_gate"]    = ((df["avg_position"] > 0) & (df["avg_position"] <= 20)).astype(int)
df["low_ctr_score"]    = percentile_rank(-df["ctr"])   # higher score = lower CTR

# Readable on purpose: visibility x "is it even seen" x "how bad is the CTR"
df["baseline_action_score"] = (df["visibility_score"] * df["position_gate"] * df["low_ctr_score"]).round(4)

df["reason_code"] = "low_ctr_visible_page"
score_thresh = df["baseline_action_score"].quantile(0.90)
df["suggested_action"] = np.where(
    df["baseline_action_score"] >= score_thresh, "refresh_and_review_ctr", "monitor"
)
df["baseline_rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

out_cols = ["content_id", "client_id", "baseline_rank", "baseline_action_score", "reason_code",
            "suggested_action", "impressions_90d", "avg_position", "ctr",
            "days_since_last_update", "is_declining_label"]
queue = df[out_cols].sort_values("baseline_rank").reset_index(drop=True)

out_path = Path("../outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)
print(f"Wrote {len(queue):,} rows -> {out_path}")

# The receipts -- this JSON is what stays committed, not the CSV itself.
import json as _json
metrics = {
    "rows": int(len(queue)),
    "score_90th_pct_threshold": float(score_thresh),
    "declining_rate_overall": float(df["is_declining_label"].mean()),
    "declining_rate_top10": float(queue.head(10)["is_declining_label"].mean()),
    "declining_rate_top50": float(queue.head(50)["is_declining_label"].mean()),
    "signal_verdicts": {"staleness_vs_decline": "MIXED", "ctr_vs_decline_top20": "CONFIRMED"},
    "score_formula": "visibility_score * position_gate * low_ctr_score",
    "reason_code": "low_ctr_visible_page",
}
metrics_path = Path("../outputs/baseline_metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "w") as f:
    _json.dump(metrics, f, indent=2)
print(f"Wrote metrics -> {metrics_path}")
print(metrics)


Wrote 30,000 rows -> ../outputs/baseline_action_score.csv
Wrote metrics -> ../outputs/baseline_metrics.json
{'rows': 30000, 'score_90th_pct_threshold': 0.346, 'declining_rate_overall': 0.5420666666666667, 'declining_rate_top10': 0.6, 'declining_rate_top50': 0.78, 'signal_verdicts': {'staleness_vs_decline': 'MIXED', 'ctr_vs_decline_top20': 'CONFIRMED'}, 'score_formula': 'visibility_score * position_gate * low_ctr_score', 'reason_code': 'low_ctr_visible_page'}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top10 = queue.head(10)
print(top10.to_string(index=False))


          content_id         client_id  baseline_rank  baseline_action_score          reason_code       suggested_action  impressions_90d  avg_position  ctr  days_since_last_update  is_declining_label
content_c8e9d6ab9013 client_19581e27de              1                 0.7791 low_ctr_visible_page refresh_and_review_ctr           208678           9.7  0.0                     104                   1
content_f986bd514b6e client_7f2253d7e2              2                 0.7397 low_ctr_visible_page refresh_and_review_ctr            22456           6.6  0.0                      20                   1
content_ae6d1339904d client_7f2253d7e2              3                 0.7259 low_ctr_visible_page refresh_and_review_ctr            17622          19.5  0.0                      20                   1
content_825a9788af8d client_4e07408562              4                 0.7231 low_ctr_visible_page refresh_and_review_ctr            16786           5.6  0.0                     104                

**One line each — action / why / what would make it wrong:**

1. `content_c8e9d6ab9013` — refresh_and_review_ctr; 208,678 impressions at position 9.7 with ~0%
   CTR is a huge gap. *Wrong if:* the near-zero CTR is a tracking/tagging artifact rather than a
   real user-behavior problem — worth a manual GSC check before assuming it's the page's fault.
2. `content_f986bd514b6e` — refresh_and_review_ctr; strong position (6.6) with real volume,
   same near-zero CTR pattern. *Wrong if:* this is a very recently updated page (20 days) still
   accumulating impression history — the ratio could still be settling.
3. `content_ae6d1339904d` — refresh_and_review_ctr; position 19.5 is right at my top-20 gate edge.
   *Wrong if:* it's borderline enough that a small ranking fluctuation would drop it out of
   scope entirely — a fragile pick.
4. `content_825a9788af8d` — refresh_and_review_ctr; solid position (5.6), real volume.
   *Wrong if:* the title/snippet is intentionally non-clickbait (e.g. a legal or medical page) —
   low CTR could be by design, not a fixable flaw.
5. `content_8ba781dafa55` — refresh_and_review_ctr; position 9.0, consistent with the pattern.
   *Wrong if:* it's a duplicate/near-duplicate of another ranking page cannibalizing its own
   clicks — the fix would be consolidation, not a rewrite.
6. `content_5d5653c4eb4f` — refresh_and_review_ctr; **not actually declining** (`is_declining_label=0`)
   despite the high score — a genuine weak pick, flagged again in Section 4.
7. `content_847a841969a2` — refresh_and_review_ctr; also **not declining**. *Wrong because:* my
   score has no decline signal in it at all — it only measures visibility + low CTR, so it will
   always surface some stable-but-low-CTR pages alongside true decliners.
8. `content_c82bc0c24241` — refresh_and_review_ctr; only 8 days since update yet still flagged —
   *Wrong if:* the CTR estimate is noisy this early after a change and just hasn't stabilized.
9. `content_eb1510f4b5f1` — refresh_and_review_ctr; **not declining**. Same root cause as #6/#7 —
   the score rewards low CTR regardless of trend.
10. `content_3e79eaafc89d` — refresh_and_review_ctr; **not declining**. Fourth of ten picks where
    low CTR alone didn't mean decline — worth remembering before trusting this score in isolation.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


**Flawed pics:** 4 out of the 10 best (`content_5d5653c4eb4f`, `content_847a841969a2`,
`content_eb1510f4b5f1`, `content_3e79eaafc89d`) **aren't** really declining. This is correct behavior, not a problem – my score only reflects visibility + low CTR without any decline signal in it whatsoever since staleness (my other decline-correlated metric) came out as MIXED in Section 1. Honest interpretation: my model identifies "pages that are visible but have under-performing CTR" - this isn't quite the same as decline but similar enough. Top-10 declining rate (0.60) versus base rate (0.54) is a clear improvement, but it's mild; top-50 rate (0.78) is more significant – the rule performs well in the top and not so well at #10.

**Leakage check:** verify that the score hasn't influenced

In [ ]:
score_inputs = {"visibility_score", "position_gate", "low_ctr_score"}
banned = {"is_declining_label", "trend_direction", "trend_pct"}
assert score_inputs.isdisjoint(banned)
print("Confirmed: baseline_action_score is built only from impressions_90d, avg_position, and ctr.")
print("None of is_declining_label / trend_direction / trend_pct feed the score or the reason code.")
print("No forward-looking window exists in this slice beyond the pre-aggregated *_90d columns, "
      "which are themselves the observation window the label is also drawn from -- not a separate "
      "future period -- so there's no additional future-window leak to check here.")


Confirmed: baseline_action_score is built only from impressions_90d, avg_position, and ctr.
None of is_declining_label / trend_direction / trend_pct feed the score or the reason code.
No forward-looking window exists in this slice beyond the pre-aggregated *_90d columns, which are themselves the observation window the label is also drawn from -- not a separate future period -- so there's no additional future-window leak to check here.


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.